In [ ]:
!pip install pystac-client planetary-computer stackstac rioxarray \
            geopandas shapely xarray numpy pandas -q


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, '/kaggle/input/farmtrust-sits-code/farmtrust_core')
sys.path.insert(0, '/kaggle/input/farmtrust-sits-code')

for farmtrust_core_path in Path('/kaggle/input').rglob('farmtrust_core'):
    sys.path.insert(0, str(farmtrust_core_path.parent))

print('FarmTrust import paths configured')


In [ ]:
START_DATE = '2022-01-01'
END_DATE = '2024-06-30'
MAX_CLOUD = 30


def bbox_to_polygon(bbox):
    min_lon, min_lat, max_lon, max_lat = bbox
    return {
        'type': 'Polygon',
        'coordinates': [
            [
                [min_lon, min_lat],
                [max_lon, min_lat],
                [max_lon, max_lat],
                [min_lon, max_lat],
                [min_lon, min_lat],
            ]
        ],
    }


AOIS = [
    {
        'aoi_id': 'aoi_nile_delta_01',
        'governorate': 'Kafr el-Sheikh governorate',
        'bbox': [30.9387, 31.1053, 30.9487, 31.1153],
    },
    {
        'aoi_id': 'aoi_nile_delta_02',
        'governorate': 'Gharbia governorate',
        'bbox': [30.8512, 30.9821, 30.8612, 30.9921],
    },
    {
        'aoi_id': 'aoi_nile_delta_03',
        'governorate': 'Beheira governorate',
        'bbox': [30.3421, 30.8234, 30.3521, 30.8334],
    },
    {
        'aoi_id': 'aoi_nile_delta_04',
        'governorate': 'Dakahlia governorate',
        'bbox': [31.3621, 31.0543, 31.3721, 31.0643],
    },
    {
        'aoi_id': 'aoi_nile_delta_05',
        'governorate': 'Sharqia governorate',
        'bbox': [31.6234, 30.7321, 31.6334, 30.7421],
    },
    {
        'aoi_id': 'aoi_upper_egypt_01',
        'governorate': 'Minya governorate',
        'bbox': [30.7512, 28.1234, 30.7612, 28.1334],
    },
    {
        'aoi_id': 'aoi_upper_egypt_02',
        'governorate': 'Assiut governorate',
        'bbox': [31.1823, 27.2341, 31.1923, 27.2441],
    },
    {
        'aoi_id': 'aoi_upper_egypt_03',
        'governorate': 'Sohag governorate',
        'bbox': [31.6934, 26.5521, 31.7034, 26.5621],
    },
    {
        'aoi_id': 'aoi_fayoum_01',
        'governorate': 'Fayoum governorate',
        'bbox': [30.8423, 29.3012, 30.8523, 29.3112],
    },
    {
        'aoi_id': 'aoi_fayoum_02',
        'governorate': 'Fayoum governorate',
        'bbox': [30.9123, 29.2234, 30.9223, 29.2334],
    },
]

for aoi in AOIS:
    aoi['geometry'] = bbox_to_polygon(aoi['bbox'])

print(f'Defined {len(AOIS)} AOIs from {START_DATE} to {END_DATE}, max_cloud={MAX_CLOUD}')


In [ ]:
from pathlib import Path
import traceback

try:
    from farmtrust_core.ingest.pipeline import run_ingest
except ImportError:
    from farmtrust_core.ingest.pipeline import write_outputs as run_ingest


WORKING_ROOT = Path('/kaggle/working')
succeeded = []
failed = []

for index, aoi in enumerate(AOIS, start=1):
    aoi_id = aoi['aoi_id']
    output_dir = WORKING_ROOT / aoi_id
    print(f'[{index}/{len(AOIS)}] INGEST {aoi_id} - {aoi["governorate"]}', flush=True)
    try:
        run_ingest(
            output_dir=output_dir,
            aoi_id=aoi_id,
            bbox=aoi['bbox'],
            start_date=START_DATE,
            end_date=END_DATE,
            max_cloud=float(MAX_CLOUD),
            force_rerun=False,
            limit_items=None,
            log_signed_hrefs=False,
            deduplicate=True,
            max_workers=4,
            geometry=aoi['geometry'],
        )
        csv_path = output_dir / 'indices_timeseries.csv'
        if not csv_path.exists():
            raise FileNotFoundError(f'Missing expected output: {csv_path}')
        print(f'SUCCESS {aoi_id} {csv_path}', flush=True)
        succeeded.append(aoi_id)
    except Exception as exc:
        print(f'FAILED {aoi_id}: {type(exc).__name__}: {exc}', flush=True)
        traceback.print_exc()
        failed.append(aoi_id)

print('Ingest summary')
print(f'  succeeded: {len(succeeded)}')
print(f'  failed: {len(failed)}')
print(f'  succeeded AOIs: {succeeded}')
print(f'  failed AOIs: {failed}')


In [ ]:
!zip -r /kaggle/working/egypt_aoi_timeseries.zip /kaggle/working/aoi_*/
import os

size = os.path.getsize('/kaggle/working/egypt_aoi_timeseries.zip') / 1e6
print(f'Output zip: {size:.1f} MB')


In [ ]:
print('=== INGEST COMPLETE. Download: egypt_aoi_timeseries.zip ===')
